In [ ]:
import pandas as pd
from astropy.table import Table
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from astropy.stats import sigma_clip
from clean_lc_paper_1 import *
from basics import *
import numpy as np
from calc_p_extrema_methods import *

In [ ]:
def calc_P_extrema_simple(time, flux):
    time = np.asarray(time)
    flux = np.asarray(flux)

    # Remove NaNs
    valid = np.isfinite(time) & np.isfinite(flux)
    time = time[valid]
    flux = flux[valid]

    # Point-to-point differences
    dflux = np.diff(flux)

    # Iterative 4-sigma clipping
    clipped = sigma_clip(
        dflux,
        sigma=4.0,
        maxiters=None,
        masked=True
    )

    # Sign of each difference; retain masked positions
    signs = np.sign(clipped.data)
    good = ~clipped.mask & (signs != 0) #mask of good data points (i.e. not removed by sigma clipping) 

    # Count crossings only between adjacent unmasked differences
    adjacent_good = good[:-1] & good[1:]
    crossings = (
        adjacent_good &
        (signs[:-1] != signs[1:])
    )

    N_extrema = np.sum(crossings)

    if N_extrema == 0:
        return np.nan

    N = len(flux)
    dt = 29.4/(24 * 60)  # days

    return 2 * N * dt / N_extrema

In [ ]:
def smooth(flux, width_days=2):
    cadence = 29.4/(60 * 24) #cadence in days
    points_per_day = int(1 / cadence)

    filter_width = width_days * points_per_day  # 10 day high pass filter

    highly_smoothed_flux1 = convolve(flux, Box1DKernel(filter_width), boundary='extend')
    smoothed = convolve(highly_smoothed_flux1, Box1DKernel(filter_width), boundary='extend')

    return smoothed

In [ ]:
#zach claytor RGs (have a measured period for these)
zc = pd.read_csv('../data/claytor_rgs.csv')
test = zc[zc['kbonus_designation']=='kic-003560935'] #chosen because it has nice oscillations and no nastiness. numax = 1.6 muHz

In [ ]:
fname = test['kbonus_designation'].iloc[0]
num_id = fname.split('-')[-1]
P_zc = 1000000/(test['numax'].iloc[0] * 60 * 60 * 24) #x10^6 to convert to Hertz, then take reciprocal to get period in days

## Full LC

In [ ]:
#open fits file and download lightcurve quarters
filepath = f'/Users/z5214708/tess_data/stello/Kepler/LC/KBonus/kbonus-bkg/lcs/{num_id[0:4]}/{num_id}/hlsp_kbonus-bkg_kepler_kepler_{fname}_kepler_v1.0_lc.fits'
stitched = Table.read(filepath, hdu=1, format='fits', unit_parse_strict='silent')

#remove NaNs
nan_mask = np.isnan(stitched['SAP_FLUX'])
time = np.asarray(stitched['TIME'][~nan_mask])
flux = np.asarray(stitched['SAP_FLUX'][~nan_mask])

#split into quarters
time_q, flux_q = split_light_curve(time,flux)

#calculate p extrema per quarter
p_extrema_vals = []
for i in range(len(time_q)):
    p_extrema_vals.append(calc_P_extrema_simple(time_q[i],flux_q[i]))

# Concatenate all quarters into one light curve
time = np.concatenate(time_q)
flux = np.concatenate(flux_q)

p_extrema_tot = calc_P_extrema_simple(time,flux)

# Normalize the p_extrema values to [0, 1] for colormap mapping
norm = Normalize(vmin=min(p_extrema_vals), vmax=max(p_extrema_vals))
cmap = cm.RdYlBu

#plot lc with each quarter coloured by its P_extrema val
plt.figure(figsize=(12, 8))
for i in range(len(time_q)):
    color = cmap(norm(p_extrema_vals[i]))
    plt.scatter(time_q[i], flux_q[i], s=1, color=color)

# Add a colorbar to show the mapping
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca())
cbar.set_label('P_extrema (days)')

plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.title(f'P_ZC = {P_zc:.2f} days. P_extrema_tot = {p_extrema_tot:.2f} days.')
plt.savefig('../figures/calc_p_extrema/3560935_p_ext_simple.png',bbox_inches='tight')

# One Q

In [ ]:
#pick a random quarter
time = time_q[4]
flux = flux_q[4]

p_extrema = calc_P_extrema_simple(time,flux)

# Remove NaNs
valid = np.isfinite(time) & np.isfinite(flux)
time = time[valid]
flux = flux[valid]

# Point-to-point differences
dflux = np.diff(flux)


# Iterative 4-sigma clipping
clipped = sigma_clip(
    dflux,
    sigma=4.0,
    maxiters=None,
    masked=True
)

In [ ]:
fig,axs = plt.subplots(3,1,figsize=(12,12),sharex = True , gridspec_kw={'hspace':0.03})
#original light curve
axs[0].plot(time,flux)
axs[0].set_ylabel('Flux', fontsize=(16))
#ptp difference (should pass through 0 whenever there is a turning point... sorta like taking the derivative)
axs[1].plot(time[1:],dflux)
axs[1].set_ylabel('PTP(Flux)', fontsize=16)
#there are big jumps between quarters, which means a big jump in the ptp. sigma clip to account for this
axs[2].plot(time[1:],clipped)
axs[2].set_ylabel('PTP(FLUX) (4-$\sigma$ Clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]',fontsize=16)
plt.savefig('../figures/calc_p_extrema/3560935_q4.png',bbox_inches='tight')

In [ ]:
# Convert clipped result to arrays
dflux_values = clipped.data
dflux_mask = np.ma.getmaskarray(clipped)

# Signs of the point-to-point differences
signs = np.sign(dflux_values)

# Valid adjacent pairs
valid_pairs = (
    ~dflux_mask[:-1]
    & ~dflux_mask[1:]
    & (signs[:-1] != 0)
    & (signs[1:] != 0)
)

# Sign changes = extrema
crossing_idx = np.where(
    valid_pairs & (signs[:-1] != signs[1:])
)[0]

# Times corresponding to dflux values
dflux_time = time[1:]

# Interpolate the zero-crossing time
t1 = dflux_time[crossing_idx]
t2 = dflux_time[crossing_idx + 1]
y1 = dflux_values[crossing_idx]
y2 = dflux_values[crossing_idx + 1]

crossing_time = t1 - y1 * (t2 - t1) / (y2 - y1)

# Approximate flux at the extrema
extrema_flux = np.interp(crossing_time, time, flux)

In [ ]:
fig, axs = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    gridspec_kw={'hspace': 0.03}
)

# Light curve
axs[0].plot(time, flux, color='black', lw=0.8)
axs[0].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)
axs[0].legend()

# Unclipped differences
axs[1].plot(dflux_time, dflux, color='black', lw=0.8)
axs[1].axhline(0, color='grey', ls='--', lw=0.8)
axs[1].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

# Clipped differences
axs[2].plot(dflux_time, clipped, color='black', lw=0.8)
axs[2].axhline(0, color='grey', ls='--', lw=0.8)
axs[2].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

axs[0].set_ylabel('Flux', fontsize=16)
axs[1].set_ylabel('PTP(Flux)', fontsize=16)
axs[2].set_ylabel(r'PTP(Flux) ($4\sigma$ clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]', fontsize=16)

for ax in axs:
    ax.tick_params(labelsize=12)

plt.savefig('../figures/calc_p_extrema/3560935_q4_crossings.png',bbox_inches='tight')

# Smooth first

In [ ]:
time = time_q[4]
flux = flux_q[4]

width_days = 2

cadence = 29.4/(60 * 24) #cadence in days
points_per_day = int(1 / cadence)

filter_width = width_days * points_per_day  # 10 day high pass filter

highly_smoothed_flux1 = convolve(flux, Box1DKernel(filter_width), boundary='extend')
smoothed = convolve(highly_smoothed_flux1, Box1DKernel(filter_width), boundary='extend')


# Point-to-point differences
dflux = np.diff(smoothed)

# Iterative 4-sigma clipping
clipped = sigma_clip(
    dflux,
    sigma=4.0,
    maxiters=None,
    masked=True
)

In [ ]:
fig,axs = plt.subplots(3,1,figsize=(12,12),sharex = True , gridspec_kw={'hspace':0.03})
#original light curve
axs[0].plot(time,smoothed)
axs[0].set_ylabel('Flux', fontsize=(16))
#ptp difference (should pass through 0 whenever there is a turning point... sorta like taking the derivative)
axs[1].plot(time[1:],dflux)
axs[1].set_ylabel('PTP(Flux)', fontsize=16)
#there are big jumps between quarters, which means a big jump in the ptp. sigma clip to account for this
axs[2].plot(time[1:],clipped)
axs[2].set_ylabel('PTP(FLUX) (4-$\sigma$ Clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]',fontsize=16)
plt.savefig('../figures/calc_p_extrema/3560935_q4_smoothed.png',bbox_inches='tight')

In [ ]:
# Convert clipped result to arrays
dflux_values = clipped.data
dflux_mask = np.ma.getmaskarray(clipped)

# Signs of the point-to-point differences
signs = np.sign(dflux_values)

# Valid adjacent pairs
valid_pairs = (
    ~dflux_mask[:-1]
    & ~dflux_mask[1:]
    & (signs[:-1] != 0)
    & (signs[1:] != 0)
)

# Sign changes = extrema
crossing_idx = np.where(
    valid_pairs & (signs[:-1] != signs[1:])
)[0]

# Times corresponding to dflux values
dflux_time = time[1:]

# Interpolate the zero-crossing time
t1 = dflux_time[crossing_idx]
t2 = dflux_time[crossing_idx + 1]
y1 = dflux_values[crossing_idx]
y2 = dflux_values[crossing_idx + 1]

crossing_time = t1 - y1 * (t2 - t1) / (y2 - y1)

# Approximate flux at the extrema
extrema_flux = np.interp(crossing_time, time, flux)

In [ ]:
fig, axs = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    gridspec_kw={'hspace': 0.03}
)

# Light curve
axs[0].plot(time, smoothed, color='black', lw=0.8)
axs[0].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)
axs[0].legend()

# Unclipped differences
axs[1].plot(dflux_time, dflux, color='black', lw=0.8)
axs[1].axhline(0, color='grey', ls='--', lw=0.8)
axs[1].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

# Clipped differences
axs[2].plot(dflux_time, clipped, color='black', lw=0.8)
axs[2].axhline(0, color='grey', ls='--', lw=0.8)
axs[2].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

axs[0].set_ylabel('Flux', fontsize=16)
axs[1].set_ylabel('PTP(Flux)', fontsize=16)
axs[2].set_ylabel(r'PTP(Flux) ($4\sigma$ clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]', fontsize=16)

for ax in axs:
    ax.tick_params(labelsize=12)

plt.savefig('../figures/calc_p_extrema/3560935_q4_smoothed_crossings.png',bbox_inches='tight')

In [ ]:
calc_P_extrema_simple(time,smoothed)

In [ ]:
P_zc

# Smooth per quarter

In [ ]:
#open fits file and download lightcurve quarters
filepath = f'/Users/z5214708/tess_data/stello/Kepler/LC/KBonus/kbonus-bkg/lcs/{num_id[0:4]}/{num_id}/hlsp_kbonus-bkg_kepler_kepler_{fname}_kepler_v1.0_lc.fits'
stitched = Table.read(filepath, hdu=1, format='fits', unit_parse_strict='silent')

#remove NaNs
nan_mask = np.isnan(stitched['SAP_FLUX'])
time = np.asarray(stitched['TIME'][~nan_mask])
flux = np.asarray(stitched['SAP_FLUX'][~nan_mask])

#split into quarters
time_q, flux_q = split_light_curve(time,flux)

#calculate p extrema per quarter
p_extrema_vals = []
for i in range(len(time_q)):
    p_extrema_vals.append(calc_P_extrema_simple(time_q[i],smooth(flux_q[i])))

# Concatenate all quarters into one light curve
time = np.concatenate(time_q)
flux = np.concatenate(flux_q)

p_extrema_tot = calc_P_extrema_simple(time,smooth(flux))

# Normalize the p_extrema values to [0, 1] for colormap mapping
norm = Normalize(vmin=min(p_extrema_vals), vmax=max(p_extrema_vals))
cmap = cm.RdYlBu

#plot lc with each quarter coloured by its P_extrema val
plt.figure(figsize=(12, 8))
for i in range(len(time_q)):
    color = cmap(norm(p_extrema_vals[i]))
    plt.scatter(time_q[i], flux_q[i], s=1, color=color)

# Add a colorbar to show the mapping
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca())
cbar.set_label('P_extrema (days)')

plt.xlabel('Time [Days]')
plt.ylabel('Flux')
plt.title(f'P_ZC = {P_zc:.2f} days. P_extrema_tot = {p_extrema_tot:.2f} days.')
plt.savefig('../figures/calc_p_extrema/3560935_p_ext_smoothed.png',bbox_inches='tight')

# Q5

In [ ]:
#pick a random quarter
time = time_q[5]
flux = flux_q[5]

smoothed = smooth(flux)


# Point-to-point differences
dflux = np.diff(smoothed)

# Iterative 4-sigma clipping
clipped = sigma_clip(
    dflux,
    sigma=4.0,
    maxiters=None,
    masked=True
)

# Convert clipped result to arrays
dflux_values = clipped.data
dflux_mask = np.ma.getmaskarray(clipped)

# Signs of the point-to-point differences
signs = np.sign(dflux_values)

# Valid adjacent pairs
valid_pairs = (
    ~dflux_mask[:-1]
    & ~dflux_mask[1:]
    & (signs[:-1] != 0)
    & (signs[1:] != 0)
)

# Sign changes = extrema
crossing_idx = np.where(
    valid_pairs & (signs[:-1] != signs[1:])
)[0]

# Times corresponding to dflux values
dflux_time = time[1:]

# Interpolate the zero-crossing time
t1 = dflux_time[crossing_idx]
t2 = dflux_time[crossing_idx + 1]
y1 = dflux_values[crossing_idx]
y2 = dflux_values[crossing_idx + 1]

crossing_time = t1 - y1 * (t2 - t1) / (y2 - y1)

# Approximate flux at the extrema
extrema_flux = np.interp(crossing_time, time, flux)

In [ ]:
fig, axs = plt.subplots(
    3, 1,
    figsize=(12, 12),
    sharex=True,
    gridspec_kw={'hspace': 0.03}
)

# Light curve
axs[0].plot(time, flux, color='black', lw=0.8)
axs[0].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)
axs[0].legend()

# Light curve
axs[1].plot(time, smoothed, color='black', lw=0.8)
axs[1].scatter(
    crossing_time,
    extrema_flux,
    color='red',
    s=35,
    zorder=5,
    label='Detected extrema'
)


# Clipped differences
axs[2].plot(dflux_time, clipped, color='black', lw=0.8)
axs[2].axhline(0, color='grey', ls='--', lw=0.8)
axs[2].scatter(
    crossing_time,
    np.zeros_like(crossing_time),
    color='red',
    s=35,
    zorder=5
)

axs[0].set_ylabel('Raw Flux', fontsize=16)
axs[1].set_ylabel('Smoothed Flux', fontsize=16)
axs[2].set_ylabel(r'PTP(Flux) ($4\sigma$ clipped)', fontsize=16)
axs[2].set_xlabel('Time [Days]', fontsize=16)

for ax in axs:
    ax.tick_params(labelsize=12)

axs[0].set_title(f'P_extrema = {calc_P_extrema_simple(time,smoothed):.2f} Days',fontsize=18)

plt.savefig('../figures/calc_p_extrema/3560935_q5_smoothed_crossings.png',bbox_inches='tight')

In [ ]:
calc_P_extrema_simple(time,smoothed)